가드 함수 검사

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Fri" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w3" / "Fri"
GOLDEN_DIR = ROOT / "backend" / "tests" / "golden"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

In [19]:
import importlib
importlib.invalidate_caches() 

from app.core.exceptions import GuardTripped, RateLimited    
from app.core.guards import ALLOWED_MODELS, check_daily_limit, check_model, check_question
from app.core.config import get_settings

settings = get_settings()

def try_guard(label: str, func, arg) -> None:

    try:
        result = func(arg)
    except (GuardTripped, RateLimited) as e:
        code = getattr(e, "status_code", None)
        print(f"{label} : {type(e).__name__} ({code}) {e}")
    else:
        print(f'{label} : 통과 → "{result}"')


print("허용 모델      :", ALLOWED_MODELS)     
print()

try_guard("정상 질문     ", check_question, "부산 출장 숙박비 한도가 얼마인가요?")
try_guard("공백 세 칸    ", check_question, "   ")
try_guard("아주 긴 질문  ", check_question, "가" * (settings.max_input_chars + 1))
try_guard("허용 밖 모델  ", check_model, "claude-opus-4-1")
try_guard("한도 초과     ", check_daily_limit, settings.daily_call_limit)

허용 모델      : {'claude-haiku-4-5'}

정상 질문      : 통과 → "부산 출장 숙박비 한도가 얼마인가요?"
공백 세 칸     : GuardTripped (400) 질문을 더 길게 해주세요.
아주 긴 질문   : GuardTripped (400) 현재 2001자. 2000자 이내로 질문을 줄여주세요.
허용 밖 모델   : GuardTripped (400) claude-opus-4-1은 허용되지 않은 모델임
한도 초과      : RateLimited (429) 오늘 호출 한도(200회)를 모두 소진함


스키마 검사

In [20]:
from pydantic import ValidationError
from app.schemas.chat import AnswerOut

CASES = [
    ("ⓐ 필드 누락  ", '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'),
    ("ⓑ 타입 어긋남", '{"answer": "1박 70,000원 이내입니다.", "sources": "DOC-HR-014", "enough_evidence": true}'),
    ("ⓒ 잡담이 앞에", '네, 알겠습니다!\n{"answer": "1박 70,000원 이내입니다.", "sources": [], "enough_evidence": false}'),
]

for label, raw in CASES:
    try:
        AnswerOut.model_validate_json(raw)
        print(f'{label}:passed')

    except ValidationError as e:
        first=e.errors(include_url=False)[0]
        print(f"{label} : loc={str(first['loc'])} / type={first['type']} / {first['msg']}")

ⓐ 필드 누락   : loc=('sources',) / type=missing / Field required
ⓑ 타입 어긋남 : loc=('sources',) / type=list_type / Input should be a valid array
ⓒ 잡담이 앞에 : loc=() / type=json_invalid / Invalid JSON: expected value at line 1 column 1


In [21]:
import importlib
import app.schemas.chat as chat_schema

chat_schema = importlib.reload(chat_schema)

print('AskOut 필드 :', list(chat_schema.AskOut.model_fields))

sample = chat_schema.AskOut(run_id='RUN-8821', answer='1박 70,000원 이내입니다.', sources=[], enough_evidence=False)
print(f'기본값      : attempts={sample.attempts} · fallback_used={sample.fallback_used}')

AskOut 필드 : ['answer', 'sources', 'enough_evidence', 'run_id', 'attempts', 'fallback_used']
기본값      : attempts=1 · fallback_used=False


In [22]:
import importlib
import inspect

importlib.invalidate_caches()    

from app.services import chat_service

print("ask 시그니처 :", inspect.signature(chat_service.ask, eval_str=True))
print(f"MAX_ATTEMPTS : {chat_service.MAX_ATTEMPTS}   (첫 호출 1 + 재시도 2)")


ask_body = inspect.getsource(chat_service.ask)

caught_names = []                                 # 잡히는 예외 이름 모아주는 리스트
for line in ask_body.splitlines():
    stripped = line.strip()                     
    if not stripped.startswith("except "):
        continue                                

    after_except = stripped[len("except "):]      
    exception_name = after_except.split(" as ")[0]  
    caught_names.append(exception_name.rstrip(":")) 

if len(caught_names) == 1:
    how_many = "하나"
else:
    how_many = f"{len(caught_names)}개"
print("잡는 예외    :", " · ".join(caught_names), how_many)

ask 시그니처 : (*, question: str, run_id: str = 'RUN-000') -> app.schemas.chat.AskOut
MAX_ATTEMPTS : 3   (첫 호출 1 + 재시도 2)
잡는 예외    : ValidationError 하나


In [23]:
import json
# 잘된 응답 
GOOD = json.dumps(
    {
        "answer": "국내출장 여비 규정 제12조에 따르면 숙박비는 1박 70,000원 이내입니다.",
        "sources": [
            {
                "doc_id": "DOC-HR-014",
                "title": "국내출장 여비 규정",
                "version": "v2.0",
                "locator": "제12조",
            }
        ],
        "enough_evidence": True,
    },
    ensure_ascii=False,
)
# 잘못된 응답 
NO_SOURCES = '{"answer": "1박 70,000원 이내입니다.", "enough_evidence": true}'
CHITCHAT = '죄송합니다. 지금은 답변을 드릴 수 없습니다.'

def next_answer(replies: list[str], attempt: int) -> str:
    if attempt <= len(replies):
        return replies[attempt - 1]
    return replies[-1]
print('준비한 응답 :', 'GOOD · NO_SOURCES · CHITCHAT 세 벌')


준비한 응답 : GOOD · NO_SOURCES · CHITCHAT 세 벌


In [ ]:
import app.integrations.factory as factory
from app.integrations.ports import LLMResult

REAL_GET_LLM = factory.get_llm

class StubLLM:
    # 정해둔 응답들을 순서대로 돌려주는 가짜 LLM 어댑터
    def __init__(self, replies: list[str]):
        self.replies = replies
        self.calls = 0

    def answer(self, *, question: str, context: list[dict], user: dict) -> LLMResult:
        self.calls += 1
        text = next_answer(self.replies, self.calls)
        return LLMResult(text=text, model='stub')

print('StubLLM 준비 완료')

In [24]:
from pydantic import ValidationError
from app.integrations.llm_claude import _extract_json
from app.schemas.chat import AnswerOut, AskOut
from app.services.chat_service import MAX_ATTEMPTS, _fallback, _hint_from

def run_loop(replies: list[str], run_id: str='RUN-0000') -> AskOut:
    hint = ''
    for attempt in range(1, MAX_ATTEMPTS + 1):
        raw = next_answer(replies, attempt)
        try:
            data = _extract_json(raw)
            AnswerOut.model_validate(data)
        except ValidationError as exc:
            hint = _hint_from(exc.errors(include_url=False))
            continue
        return AskOut(**data, run_id=run_id, attempts=attempt, fallback_used=False)
    return _fallback(run_id, MAX_ATTEMPTS)
CASES_RETRY = [('① 한 번에 성공  ', [GOOD]), ('② 한 번 실패    ', [NO_SOURCES, GOOD]), ('③ 전부 실패     ', [CHITCHAT, CHITCHAT, CHITCHAT])]
for (label, replies) in CASES_RETRY:
    out = run_loop(replies)
    source_count = len(out.sources)
    print(f'{label}{out.run_id}  attempts={out.attempts}  fallback={str(out.fallback_used):<6} 근거 {source_count}건')
    if out.fallback_used:
        print(f'   ⚠️ 폴백으로 응답합니다 (attempts={out.attempts})')

① 한 번에 성공  RUN-0000  attempts=1  fallback=False  근거 1건
② 한 번 실패    RUN-0000  attempts=2  fallback=False  근거 1건
③ 전부 실패     RUN-0000  attempts=3  fallback=True   근거 0건
   ⚠️ 폴백으로 응답합니다 (attempts=3)


In [25]:
CASE_N01 = {
    "id": "N-01",
    "kind": "N",
    "question": "부산 출장 2박 3일인데 숙박비 한도가 얼마인가요?",
    "stub": [ 
        '{"answer": "부산 출장 숙박비는 1박 7만원 이내입니다.", '
        '"sources": [{"doc_id": "DOC-HR-014", "title": "국내출장 여비 규정", '
        '"version": "v2.0", "locator": "제12조(숙박비) · p.6"}], '
        '"enough_evidence": true}'
    ],
    "expect": {
        "contains": ["7만"],              # 핵심 숫자 하나가 답에 있어야 한다
        "min_sources": 1,                 # 근거가 1건 이상
        "doc_ids": ["DOC-HR-014"],        # 이 문서에서 나와야 한다
    },
}

CASE_E01 = {
    "id": "E-01",
    "kind": "E",
    "question": "화성 출장 숙박비 한도는 얼마인가요?",
    "stub": [
        '{"answer": "국내출장 여비 규정에는 해당 지역의 숙박비 한도가 없습니다. '
        '담당 부서에 문의해 주세요.", "sources": [], "enough_evidence": false}'
    ],
    "expect": {
        "max_sources": 0,                 # 근거를 달면 안 된다
        "enough_evidence": False,         # 스스로 「근거가 부족하다」고 말해야 한다
        "not_contains": ["7만"],          # 다른 지역 숫자를 끌어다 지어내면 실패
    },
}

In [26]:
import json
from collections import Counter
GOLDEN_PATH = GOLDEN_DIR / 'chat_golden.json'
GOLDEN = json.loads(GOLDEN_PATH.read_text(encoding='utf-8'))
counts = Counter((case['kind'] for case in GOLDEN))
KIND_LABELS = {'N': 'N (정상)      ', 'E': 'E (근거 없음) ', 'F': 'F (스키마 위반)', 'G': 'G (가드)      '}
print(f'문항 {len(GOLDEN)}개')
for (kind, label) in KIND_LABELS.items():
    print(f'  {label} {counts[kind]}')
print('네 유형 모두 있는가 :', all((counts[kind] > 0 for kind in KIND_LABELS)))

NameError: name 'GOLDEN_DIR' is not defined

In [ ]:
# 1.
from types import SimpleNamespace  


def check(case: dict, out) -> list[str]:
    expect = case["expect"]
    problems: list[str] = []

    for needle in expect.get("contains", []):
        if needle not in out.answer:
            problems.append(f"answer 에 '{needle}' 이 없습니다")

    for needle in expect.get("not_contains", []):
        if needle in out.answer:
            problems.append(f"answer 에 '{needle}' 이 들어 있습니다")

    # 필드
    if "min_sources" in expect and len(out.sources) < expect["min_sources"]:
        problems.append(
            f"근거가 {expect['min_sources']}건 이상이어야 합니다 (현재 {len(out.sources)}건)"
        )
    if "max_sources" in expect and len(out.sources) > expect["max_sources"]:
        problems.append(
            f"sources 가 {expect['max_sources']}건이어야 합니다 (현재 {len(out.sources)}건)"
        )
    for key in ("attempts", "fallback_used", "enough_evidence"):
        if key in expect and getattr(out, key) != expect[key]:
            problems.append(f"{key} 가 {expect[key]} 여야 합니다")

    # 화이트리스트 검사
    if "doc_ids" in expect:
        for source in out.sources:
            if source.doc_id not in expect["doc_ids"]:
                problems.append(f"허용되지 않은 doc_id: {source.doc_id}")

    return problems

# 판정 함수에 전달할 가짜 응답 객체
def fake_out(answer: str, doc_ids: list[str], **fields) -> SimpleNamespace:
    base = {"enough_evidence": True, "attempts": 1, "fallback_used": False}
    base.update(fields)
    return SimpleNamespace(
        answer=answer,
        sources=[SimpleNamespace(doc_id=doc_id) for doc_id in doc_ids],
        **base,
    )

In [28]:
# 2.
out_ok = fake_out("부산 출장 숙박비는 1박 7만원 이내입니다.", ["DOC-HR-014"])

out_no_number = fake_out("부산 출장 숙박비는 규정에 정해진 한도를 따릅니다.", ["DOC-HR-014"])

out_fake_doc = fake_out("부산 출장 숙박비는 1박 7만원 이내입니다.", ["DOC-HR-999"])

problems_ok = check(CASE_N01, out_ok)
problems_no_number = check(CASE_N01, out_no_number)
problems_fake_doc = check(CASE_N01, out_fake_doc)

print("정상 응답        :", json.dumps(problems_ok, ensure_ascii=False))
print("숫자가 어긋남    :", json.dumps(problems_no_number, ensure_ascii=False))
print("없는 문서를 인용 :", json.dumps(problems_fake_doc, ensure_ascii=False))

정상 응답        : []
숫자가 어긋남    : ["answer 에 '7만' 이 없습니다"]
없는 문서를 인용 : ["허용되지 않은 doc_id: DOC-HR-999"]


In [29]:
# 3.
import logging

from app.core.exceptions import GuardTripped
from app.core.guards import check_model


def question_of(case: dict) -> str:
    return case["question"] * case.get("repeat", 1)


def run_case(case: dict) -> list[str]:
    expect = case["expect"]

    if "raises" in expect:
        try:
            if case.get("guard") == "check_model":
                check_model(case["question"])
            else:
                chat_service.ask(question=question_of(case))
        except GuardTripped as exc:
            return [f"예외 메시지에 '{n}' 이 없습니다"
                    for n in expect.get("contains", []) if n not in str(exc)]
        return ["GuardTripped 가 나지 않았습니다"]

    stub = StubLLM(case["stub"])
    factory.get_llm = lambda: stub
    try:
        out = chat_service.ask(question=question_of(case))
    finally:
        factory.get_llm = REAL_GET_LLM     
    if stub.calls == 0:
        return ["가짜 어댑터가 한 번도 불리지 않았습니다"]
    return check(case, out)

In [30]:
# 4.
logging.disable(logging.WARNING)
try:
    results = {case["id"]: run_case(case) for case in GOLDEN}
finally:
    logging.disable(logging.NOTSET)

for kind in KIND_LABELS:
    line = [f"{case['id']} {'YES' if not results[case['id']] else 'NO'}"
            for case in GOLDEN if case["kind"] == kind]
    print("   ".join(line))

passed = sum(1 for problems in results.values() if not problems)

print()
print(f"통과율 : {passed}/{len(GOLDEN)} = {passed / len(GOLDEN) * 100:.1f}%")

for case_id, problems in results.items():
    if problems:
        print(f"  {case_id} : {json.dumps(problems, ensure_ascii=False)}")

NameError: name 'GOLDEN' is not defined